# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant JSON-LD schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # This is a mlcroissant.Metadata object

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")

## 2. Data Overview
Review available record sets and fields, referenced by their `@id`. This is important for understanding the structure of the dataset and how to access records for analysis.

In [ ]:
# Display all record sets with their @id and field information
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', 'N/A')}")
    print(f"  Description: {getattr(rs, 'description', 'N/A')}")
    print(f"  File(s): {[fo.id for fo in getattr(rs, 'file_objects', [])]}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, type: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All access is by `@id`.

Let's load the data from each available `RecordSet` into pandas DataFrames. The fields and record set `@id`s are taken directly from their definitions above.

In [ ]:
# Collect the RecordSet @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
print("RecordSet IDs:", record_set_ids)

# Load each record set into a DataFrame
dataframes = {}
for rs_id in record_set_ids:
    print(f"\nLoading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns for {rs_id}: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For illustration, we'll use the first available RecordSet and its available numeric fields (in mass spectrometry data, likely candidates are protein quantifications, coverage, or counts—check field IDs/types above and select accordingly).

In [ ]:
# Select the first RecordSet for EDA
if not record_set_ids:
    print("No available RecordSets in metadata.")
else:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]

    # Show numeric fields
    rs = next(rs for rs in metadata.record_sets if rs.id == rs_id)
    numeric_fields = [f.id for f in rs.fields if f.data_type in ['Float', 'Integer', 'Number', 'Double']]
    print(f"Numeric fields in RecordSet {rs_id}: {numeric_fields}")
    if not numeric_fields:
        print("No numeric fields available for EDA.")
    else:
        numeric_field = numeric_fields[0]

        # Filtering: show only records above a threshold (example: top 10 values)
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > mean ({threshold}):")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping: use a categorical field (choose the first that's not numeric or ID)
        possible_group_fields = [f.id for f in rs.fields if f.data_type in ['Text', 'String', 'Category'] and f.id != numeric_field]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}: average {numeric_field}")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we'll show the distribution of the selected numeric field and, if available, boxplots/grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have successfully loaded, explored, and visualized the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells] dataset using `mlcroissant` and `pandas`. The notebook demonstrates referencing of `record sets`, `fields`, and specific columns by their `@id` as defined by Croissant. You may further adapt filtering and visualization steps using alternate fields and record sets as available in the dataset.